# MediSense-Cardio -- confirmation of the straightening fix on 40 NEW photos (Kaggle, GPU)

**Why:** on the first 60 photos, straightening the page before digitising cut the failures from 20/60 to 2/60 (exploratory). This run tests the SAME fix, unchanged, on 40 new photos of 20 new pages. Rules were fixed before the photos existed (docs/plan/analysis_plan_changes.md, 2026-11-11).
**What it does:** takes the 40 new photos, makes four versions of each (V0 original, V1 enlarge, V2 straighten the page, V3 straighten and enlarge), digitises all 160 images with the same digitiser and settings, and saves the results. The comparison with the originals is done afterwards on your computer.

**Steps (Kaggle):**
1. New notebook -> File -> Import notebook -> upload this file.
2. File -> Add input -> choose your NEW dataset `ownphotos2` (the 40 new photos).
3. Settings: Accelerator = GPU, Internet = On.
4. Save Version -> Save & Run All (Commit). About 1 hour. When the Logs show `ALL PREPARED PHOTOS DIGITISED`, open Output, download the zip and tell me.


## Step 1 -- Find the photos and make the three prepared versions


In [ ]:
import os, glob, json, time, shutil, subprocess, sys, datetime
import numpy as np, pandas as pd
WORK = '/kaggle/working'; TMP = '/tmp/confirm'; INPUT_ROOT = '/kaggle/input'
DIG_DIR = f'{TMP}/Open-ECG-Digitizer'
os.makedirs(WORK, exist_ok=True); os.makedirs(TMP, exist_ok=True)
photos = sorted(p for p in glob.glob(INPUT_ROOT + '/**/*', recursive=True) if os.path.isfile(p) and p.lower().endswith(('.jpg', '.jpeg', '.png')))
print(f'{len(photos)} photo files found')
assert len(photos) == 40, 'expected the 40 photos of the dataset ownphotos2: File -> Add input -> ownphotos2'
names = [os.path.basename(p) for p in photos]
assert len(set(names)) == 40


In [ ]:
import cv2
import numpy as np


def enlarge(img, long_side=4096):
    h, w = img.shape[:2]
    if max(h, w) >= long_side:
        return img
    s = long_side / max(h, w)
    return cv2.resize(img, (int(round(w * s)), int(round(h * s))), interpolation=cv2.INTER_LANCZOS4)


def _order(pts):
    pts = np.asarray(pts, dtype=np.float32).reshape(4, 2)
    s = pts.sum(axis=1)
    d = np.diff(pts, axis=1).ravel()
    return np.array([pts[np.argmin(s)], pts[np.argmin(d)], pts[np.argmax(s)], pts[np.argmax(d)]], dtype=np.float32)      # tl, tr, br, bl


def find_page_quad(img):
    h, w = img.shape[:2]
    sc = 800.0 / max(h, w)
    small = cv2.resize(img, (int(w * sc), int(h * sc)), interpolation=cv2.INTER_AREA)
    gray = cv2.GaussianBlur(cv2.cvtColor(small, cv2.COLOR_BGR2GRAY), (7, 7), 0)
    _, th = cv2.threshold(gray, 0, 255, cv2.THRESH_BINARY + cv2.THRESH_OTSU)
    th = cv2.morphologyEx(th, cv2.MORPH_CLOSE, np.ones((15, 15), np.uint8))
    th = cv2.morphologyEx(th, cv2.MORPH_OPEN, np.ones((9, 9), np.uint8))
    cnts, _ = cv2.findContours(th, cv2.RETR_EXTERNAL, cv2.CHAIN_APPROX_SIMPLE)
    if not cnts:
        return None
    best = None
    for c in sorted(cnts, key=cv2.contourArea, reverse=True)[:3]:
        area = cv2.contourArea(c)
        if area < 0.12 * small.shape[0] * small.shape[1]:
            continue
        hull = cv2.convexHull(c)
        peri = cv2.arcLength(hull, True)
        quad = None
        for eps in (0.015, 0.02, 0.03, 0.04, 0.06):
            ap = cv2.approxPolyDP(hull, eps * peri, True)
            if len(ap) == 4:
                quad = ap.reshape(4, 2)
                break
        if quad is None:
            quad = cv2.boxPoints(cv2.minAreaRect(hull))
        q = _order(quad)
        wd = (np.linalg.norm(q[1] - q[0]) + np.linalg.norm(q[2] - q[3])) / 2
        ht = (np.linalg.norm(q[3] - q[0]) + np.linalg.norm(q[2] - q[1])) / 2
        if ht < 1 or not (0.9 < wd / ht < 1.8):
            continue
        best = q / sc
        break
    return best


def straighten(img, out_w=2200, out_h=1700):
    q = find_page_quad(img)
    if q is None:
        return img, False
    dst = np.array([[0, 0], [out_w - 1, 0], [out_w - 1, out_h - 1], [0, out_h - 1]], dtype=np.float32)
    M = cv2.getPerspectiveTransform(q.astype(np.float32), dst)
    return cv2.warpPerspective(img, M, (out_w, out_h), flags=cv2.INTER_CUBIC, borderMode=cv2.BORDER_REPLICATE), True



In [ ]:
VAR_DIR = f'{TMP}/variants'; os.makedirs(VAR_DIR, exist_ok=True)
rows = []; found = {}
for p in photos:
    stem = os.path.splitext(os.path.basename(p))[0]
    img = cv2.imread(p)
    v0 = img
    v1 = enlarge(img)
    v2, ok2 = straighten(img, 2200, 1700)
    v3, ok3 = straighten(img, 4400, 3400)
    found[stem] = bool(ok2 and ok3)
    for tag, im in (('V0_original', v0), ('V1_enlarge', v1), ('V2_straighten', v2), ('V3_straighten_big', v3)):
        path = f'{VAR_DIR}/{stem}__{tag}.png'
        cv2.imwrite(path, im)
        rows.append({'variant_id': f'{stem}__{tag}', 'photo': stem, 'variant': tag, 'source_path': path, 'page_found': found[stem] if tag in ('V2_straighten', 'V3_straighten_big') else True,
                     'img_h': im.shape[0], 'img_w': im.shape[1]})
all_variants = pd.DataFrame(rows)
all_variants.drop(columns='source_path').to_csv(f'{WORK}/confirm_variants.csv', index=False)
print(len(all_variants), 'prepared images; photos where the page was found:', sum(found.values()), 'of 40')


## Step 2 -- Install the tools and set up the digitiser (same as the main study)


In [ ]:
def sh(cmd, cwd=None, show=True):
    r = subprocess.run(cmd, shell=True, capture_output=True, text=True, cwd=cwd)
    if show and r.returncode != 0:
        print((r.stdout + r.stderr)[-1500:])
    return r

r = subprocess.run([sys.executable, '-m', 'pip', 'install', '-q', 'neurokit2', 'opencv-python-headless', 'yacs', 'scikit-image', 'tqdm', 'torch-tps'],
                   capture_output=True, text=True)
print('pip install:', 'ok' if r.returncode == 0 else 'FAILED\n' + r.stderr[-800:])
import neurokit2 as nk, cv2, torch
device = 'cuda' if torch.cuda.is_available() else 'cpu'
print('device:', device)
if device == 'cpu':
    print('WARNING: no GPU. Right panel -> Settings -> Accelerator -> choose a GPU, then Run All again. On CPU this takes about 2 minutes per image.')



In [ ]:
if os.path.exists(DIG_DIR):
    shutil.rmtree(DIG_DIR)
r = sh(f'git clone https://github.com/Ahus-AIM/Open-ECG-Digitizer.git {DIG_DIR}')
assert os.path.isdir(f'{DIG_DIR}/src'), 'git clone failed. Is Internet switched on? Right panel -> Settings -> Internet -> On.'
sh('apt-get -qq install -y git-lfs', show=False)
sh('git lfs install', cwd=DIG_DIR, show=False)
sh('git lfs pull', cwd=DIG_DIR)
path = f'{DIG_DIR}/src/utils.py'
content = open(path).read()
if 'Stopper = object' not in content:      # patch only once
    content = content.replace("from ray.tune import Stopper",
                              "try:\n    from ray.tune import Stopper\nexcept ImportError:\n    Stopper = object  # ray not needed for inference")
    open(path, 'w').write(content)
wts = {f: os.path.getsize(f'{DIG_DIR}/weights/{f}') for f in os.listdir(f'{DIG_DIR}/weights')} if os.path.isdir(f'{DIG_DIR}/weights') else {}
print('weights:', wts)
assert wts.get('unet_weights_07072025.pt', 0) > 1e6 and wts.get('lead_name_unet_weights_07072025.pt', 0) > 1e6, \
    'The digitiser weight files are missing or only pointers (git-lfs problem): paste the messages above into the chat.'
print('Digitiser ready.')



In [ ]:
def write_digitizer_config(images_path, output_path):
    config_content = f'''MODEL:
  class_path: 'src.model.inference_wrapper.InferenceWrapper'
  KWARGS:
    config:
      SIGNAL_EXTRACTOR:
        class_path: 'src.model.signal_extractor.SignalExtractor'
        KWARGS: {{}}
      PERSPECTIVE_DETECTOR:
        class_path: 'src.model.perspective_detector.PerspectiveDetector'
        KWARGS:
          num_thetas: 250
      DEWARPER:
        class_path: 'src.model.dewarper.Dewarper'
        KWARGS:
          abs_peak_threshold: 0.1
      SEGMENTATION_MODEL:
        class_path: 'src.model.unet.UNet'
        weight_path: './weights/unet_weights_07072025.pt'
        KWARGS:
          num_in_channels: 3
          num_out_channels: 4
          dims: [32, 64, 128, 256, 320, 320, 320, 320]
          depth: 2
      CROPPER:
        class_path: 'src.model.cropper.Cropper'
        KWARGS:
          granularity: 80
          percentiles: [0.02, 0.98]
          alpha: 0.85
      PIXEL_SIZE_FINDER:
        class_path: 'src.model.pixel_size_finder.PixelSizeFinder'
        KWARGS:
          min_number_of_grid_lines: 30
          max_number_of_grid_lines: 70
          lower_grid_line_factor: 0.5
      LAYOUT_IDENTIFIER:
        class_path: 'src.model.lead_identifier.LeadIdentifier'
        config_path: 'src/config/lead_layouts_george-moody-2024.yml'
        unet_config_path: 'src/config/lead_name_unet.yml'
        unet_weight_path: './weights/lead_name_unet_weights_07072025.pt'
        KWARGS:
          debug: false
          device: '{device}'
          possibly_flipped: false
          target_num_samples: 10000
          required_valid_samples: 2
    device: '{device}'
    resample_size: 3000
    rotate_on_resample: true
    enable_timing: false
    apply_dewarping: false
DATA:
  images_path: '{images_path}'
  image_extensions: ['.png', '.jpg', '.jpeg', '.JPG']
  save_mode: 'all'
  layout_should_include_substring: false
  output_path: '{output_path}'
  clear_output_dir_if_exists: true
'''
    with open(f'{DIG_DIR}/batch_config.yml', 'w') as f:
        f.write(config_content)

print("Digitizer config helper ready (same settings as the main study).")



In [ ]:
FS_CANONICAL = 1000.0
OUT_NPZ = f'{WORK}/confirm_digitized_12lead'; os.makedirs(OUT_NPZ, exist_ok=True)
RESULTS_PATH = f'{WORK}/results_confirm.csv'
def run_batch(batch):
    imgs, outd = f'{TMP}/images_batch', f'{TMP}/digitized_batch'
    for d in (imgs, outd):
        if os.path.exists(d): shutil.rmtree(d)
    os.makedirs(imgs)
    for _, v in batch.iterrows():
        shutil.copy(v['source_path'], f'{imgs}/{v["variant_id"]}.png')
    write_digitizer_config(imgs, outd)
    t0 = time.time()
    r = subprocess.run([sys.executable, '-m', 'src.digitize', '--config', f'{DIG_DIR}/batch_config.yml'], cwd=DIG_DIR, capture_output=True, text=True)
    rows = []
    for _, v in batch.iterrows():
        vid = v['variant_id']; csv_path = f'{outd}/{vid}_timeseries_canonical.csv'
        row = {'variant_id': vid, 'has_output': False, 'n_II_samples': 0}
        if os.path.exists(csv_path):
            dg = pd.read_csv(csv_path)
            np.savez_compressed(f'{OUT_NPZ}/{vid}.npz', **{c: dg[c].to_numpy(dtype='float32') for c in dg.columns})
            row['has_output'] = True
            if 'II' in dg.columns: row['n_II_samples'] = int(dg['II'].notna().sum())
        rows.append(row)
    return pd.DataFrame(rows), time.time() - t0, (r.stdout + r.stderr)
results_so_far = pd.read_csv(RESULTS_PATH) if os.path.exists(RESULTS_PATH) else pd.DataFrame()
done = set(results_so_far['variant_id']) if len(results_so_far) else set()
remaining = all_variants[~all_variants['variant_id'].isin(done)].reset_index(drop=True)
print(f'{len(all_variants)} images in total, {len(remaining)} still to do.')


## Step 3 -- Digitise all 160 prepared images (checkpointed; smoke test first)


In [ ]:
smoke, secs, tool_out = run_batch(remaining.iloc[:2].reset_index(drop=True))
print(smoke.to_string()); print(f'smoke test took {secs:.0f} s for 2 images')
if not smoke['has_output'].any() and ('Traceback' in tool_out or 'Error' in tool_out[-1800:]):
    print(tool_out[-1800:]); raise SystemExit('STOPPED on purpose: paste the output above into the chat.')
SUBBATCH_SIZE = 10; SESSION_TIME_BUDGET_S = 5 * 3600; session_start = time.time()
while len(remaining) > 0 and (time.time() - session_start) < SESSION_TIME_BUDGET_S:
    batch = remaining.iloc[:SUBBATCH_SIZE].reset_index(drop=True)
    br, secs, _ = run_batch(batch)
    results_so_far = pd.concat([results_so_far, br], ignore_index=True).drop_duplicates(subset='variant_id', keep='last')
    tmp_path = RESULTS_PATH + '.tmp'; results_so_far.to_csv(tmp_path, index=False); os.replace(tmp_path, RESULTS_PATH)
    assert len(pd.read_csv(RESULTS_PATH)) == len(results_so_far), 'SAVE VERIFY FAILED'
    remaining = remaining.iloc[SUBBATCH_SIZE:].reset_index(drop=True)
    print(f'Checkpointed {len(batch)} images ({secs/len(batch):.1f} s/image). Done {len(results_so_far)}/{len(all_variants)}, {len(remaining)} remaining.')
print('ALL PREPARED PHOTOS DIGITISED. Download: results_confirm.csv, confirm_variants.csv, and the folder confirm_digitized_12lead.' if len(remaining) == 0 else f'{len(remaining)} images remaining: run this cell again.')
print(results_so_far.merge(all_variants[['variant_id', 'variant']], on='variant_id').groupby('variant').has_output.sum().to_string())
